# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NAJAM2005/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

> **Data note.** This notebook uses the public 30,000-page starter CSV in the repo (32 clients, 54.2% of pages labelled declining). The capstone uses a different, gated lane cut (10,550 pages, 29 clients, 45.0% declining, separate impression windows), so numbers here are not directly comparable with the capstone paper. Both are observed, directional and for decision-support.

**Method: gradient boosting (shallow trees, 150 rounds), compared with a transparent rule.** Why it fits: the data is tabular, skewed and partly missing (word count 25.7% missing, search volume 8.2%), and boosting handles missing values and skewed fields without extra cleaning. I did not use a neural net (30,000 rows of tabular data do not justify it) and I keep the Week-4 rule as the bar to beat. Inputs are static page facts plus update age and impressions; every column that holds the label window (last/previous 30 days, trend fields, product flags, 90-day CTR and position) is left out on purpose. One honest caveat: `impressions_90d` is a 90-day total that contains the label window, so it is a leakage risk (tested in the validation audit).

In [1]:
import os, json, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
RAW = "https://raw.githubusercontent.com/NAJAM2005/flyrank-ml-internship/main/"
PATH = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(PATH if os.path.exists(PATH) else RAW + PATH)      # public starter slice (anonymised)
df["is_declining"] = (df["trend_direction"] == "down").astype(int)   # outcome for checks only, never a feature
print(df.shape, "| declining share:", round(df["is_declining"].mean(), 3), "| clients:", df["client_id"].nunique())

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
NUM_BASE = ["days_since_last_update", "impressions_90d"]
STATIC = ["content_age_days", "word_count", "search_volume", "competition", "cpc"]
CAT = ["content_type", "main_intent"]
EXCLUDED = {"trend_direction / trend_pct": "the label is built from them",
            "impressions_last_30d / prev_30d (and clicks, sessions)": "they hold the window the label is computed from",
            "ctr, avg_position, engagement, scroll": "90-day aggregates that include the label window",
            "health_score and product flags": "FlyRank composite and flags; leakage-risk context, not features"}
print("model inputs (baseline):", NUM_BASE)
print("model inputs (+ static):", STATIC + CAT)
print("\nexcluded on purpose:")
for k, v in EXCLUDED.items():
    print(f"  - {k}: {v}")
print("\nmissing share in static inputs:", df[STATIC].isna().mean().round(3).to_dict())
print("Why boosting: handles missing values and skewed fields natively; compared with a transparent rule below.")


(30000, 45) | declining share: 0.542 | clients: 32


model inputs (baseline): ['days_since_last_update', 'impressions_90d']
model inputs (+ static): ['content_age_days', 'word_count', 'search_volume', 'competition', 'cpc', 'content_type', 'main_intent']

excluded on purpose:
  - trend_direction / trend_pct: the label is built from them
  - impressions_last_30d / prev_30d (and clicks, sessions): they hold the window the label is computed from
  - ctr, avg_position, engagement, scroll: 90-day aggregates that include the label window
  - health_score and product flags: FlyRank composite and flags; leakage-risk context, not features

missing share in static inputs: {'content_age_days': 0.0, 'word_count': 0.257, 'search_volume': 0.082, 'competition': 0.082, 'cpc': 0.082}
Why boosting: handles missing values and skewed fields natively; compared with a transparent rule below.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split: grouped by client, 5 folds.** The real use is ranking pages for a client the model has not seen, so no client may appear in both training and test. The table shows 0 clients in both for every fold. Fold 0 is a single large client (7,008 pages); the other folds hold 7 to 8 clients, and test decline rates range from 37.9% to 64.5%, so single-fold results are noisy. A time-aware split is not possible because this slice is one snapshot.

In [2]:
from sklearn.model_selection import GroupKFold, KFold
X = df[NUM_BASE + STATIC].copy()
for c in CAT:
    X[c] = df[c].astype("category")
y = df["is_declining"]
groups = df["client_id"]
gkf = GroupKFold(n_splits=5)
rows = []
for i, (tr, te) in enumerate(gkf.split(X, y, groups)):
    overlap = set(groups.iloc[tr]) & set(groups.iloc[te])
    rows.append({"fold": i, "test_clients": groups.iloc[te].nunique(), "test_pages": len(te),
                 "test_decline_rate": round(y.iloc[te].mean(), 3), "clients_in_both_train_and_test": len(overlap)})
print(pd.DataFrame(rows).to_string(index=False))
print("\nbase rate:", round(y.mean(), 3))


 fold  test_clients  test_pages  test_decline_rate  clients_in_both_train_and_test
    0             1        7008              0.490                               0
    1             7        5731              0.645                               0
    2             8        5753              0.379                               0
    3             8        5755              0.622                               0
    4             8        5753              0.585                               0

base rate: 0.542


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Same data, same label, same split as the Week-4 baseline.** Base rate 54.2%.

| Ranker | Pooled AUC | Mean fold AUC | Precision in top 10% |
|---|---|---|---|
| Baseline rule | 0.590 | 0.596 | 0.514 |
| Boosting, same inputs | 0.609 | 0.617 | 0.544 |
| Boosting, + static features | 0.622 | 0.609 | 0.599 |

The models beat the rule in 4 of 5 folds (and lose clearly in fold 3: 0.652 / 0.644 against 0.720), so the gain is small and not consistent. With 5 folds I do not claim a reliable difference. For reference, the capstone on the lane data, evaluated per client, found no reliable model gain either (0.567 against 0.561).

In [3]:
from sklearn.metrics import roc_auc_score
df["score_baseline"] = 0.6 * (df["days_since_last_update"] > 22).astype(float) + 0.4 * np.log1p(df["impressions_90d"]).rank(pct=True)

def boost():
    return HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150, random_state=0,
                                          categorical_features="from_dtype")
def cv_scores(cols):
    s = pd.Series(np.nan, index=df.index)
    for tr, te in gkf.split(X, y, groups):
        m = boost().fit(X.iloc[tr][cols], y.iloc[tr])
        s.iloc[te] = m.predict_proba(X.iloc[te][cols])[:, 1]
    return s

df["score_model_same_inputs"] = cv_scores(NUM_BASE)
df["score_model_plus_static"] = cv_scores(NUM_BASE + STATIC + CAT)

def p_at_top(s, frac=0.10):
    k = int(len(df) * frac)
    return df.loc[s.sort_values(ascending=False).index[:k], "is_declining"].mean()

def mean_fold_auc(s):
    return np.mean([roc_auc_score(y.iloc[te], s.iloc[te]) for _, te in gkf.split(X, y, groups)])

res = []
for name, col in [("baseline rule", "score_baseline"), ("boosting, same inputs", "score_model_same_inputs"),
                  ("boosting, + static features", "score_model_plus_static")]:
    res.append({"ranker": name, "pooled_AUC": round(roc_auc_score(y, df[col]), 3),
                "mean_fold_AUC": round(mean_fold_auc(df[col]), 3), "precision_top10%": round(p_at_top(df[col]), 3)})
tab = pd.DataFrame(res)
print(tab.to_string(index=False))
print("\nbase rate:", round(y.mean(), 3), "(precision at top 10% is compared with this)")

folds = []
for i, (tr, te) in enumerate(gkf.split(X, y, groups)):
    folds.append({"fold": i, "baseline": round(roc_auc_score(y.iloc[te], df["score_baseline"].iloc[te]), 3),
                  "same_inputs": round(roc_auc_score(y.iloc[te], df["score_model_same_inputs"].iloc[te]), 3),
                  "plus_static": round(roc_auc_score(y.iloc[te], df["score_model_plus_static"].iloc[te]), 3)})
print("\nper-fold AUC:")
print(pd.DataFrame(folds).to_string(index=False))

cap = json.loads(pd.io.common.urlopen(RAW + "work/outputs/capstone_metrics.json").read())
print("\nfor reference, capstone (client-held-out, lane data) mean per-client AUC:", cap["mean_client_auc"])


                     ranker  pooled_AUC  mean_fold_AUC  precision_top10%
              baseline rule       0.590          0.596             0.514
      boosting, same inputs       0.609          0.617             0.544
boosting, + static features       0.622          0.609             0.599

base rate: 0.542 (precision at top 10% is compared with this)

per-fold AUC:
 fold  baseline  same_inputs  plus_static
    0     0.552        0.563        0.583
    1     0.465        0.550        0.518
    2     0.622        0.692        0.677
    3     0.720        0.652        0.644
    4     0.622        0.629        0.623



for reference, capstone (client-held-out, lane data) mean per-client AUC: {'baseline': 0.5606, 'age_only': 0.5298, 'old_window+age': 0.5506, 'old_window+age+static': 0.5667, '+freshness (sensitivity)': 0.5888}


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**Where it is wrong and what it leans on.**

- **By page type:** within fresh pages (updated in the last 22 days) the model separates declining pages with an AUC of 0.652; within stale pages only 0.544.
- **Calibration by score decile:** the lowest decile really is lowest risk (16.9% declining), but above the middle the curve flattens and bends down (66.5% at decile 6, 59.9% at decile 10). So the scores find low-risk pages better than they order the top.
- **What it leans on (permutation importance, one held-out fold, which is a single client, so directional):** page age dominates (a drop of 0.065 in AUC when shuffled); search volume and impressions are small; update age, word count, CPC and competition are about zero.

Short version: weak signal, mostly page age, not reliable at the top of the ranking.

In [4]:
# Where is the model wrong?  1) by page type, 2) calibration by decile, 3) what it leans on.
s = df["score_model_plus_static"]
df["stale_flag"] = np.where(df["days_since_last_update"] > 22, "stale (>22d)", "fresh (<=22d)")
rows = []
for k, g in df.groupby("stale_flag"):
    rows.append({"group": k, "n": len(g), "decline_rate": round(g["is_declining"].mean(), 3),
                 "AUC": round(roc_auc_score(g["is_declining"], s.loc[g.index]), 3)})
print(pd.DataFrame(rows).to_string(index=False))

df["decile"] = pd.qcut(s.rank(method="first"), 10, labels=False) + 1
cal = df.groupby("decile").agg(pages=("is_declining", "size"), mean_score=("score_model_plus_static", "mean"),
                               actual_decline_rate=("is_declining", "mean")).round(3)
print("\ncalibration by score decile (1 = lowest risk):")
print(cal.to_string())

from sklearn.inspection import permutation_importance
tr, te = next(iter(gkf.split(X, y, groups)))
m = boost().fit(X.iloc[tr], y.iloc[tr])
pi = permutation_importance(m, X.iloc[te], y.iloc[te], scoring="roc_auc", n_repeats=5, random_state=0)
imp = pd.Series(pi.importances_mean, index=X.columns).sort_values(ascending=False).round(4)
print("\npermutation importance on one held-out fold (drop in AUC when the column is shuffled):")
print(imp.to_string())


        group     n  decline_rate   AUC
fresh (<=22d) 19430         0.512 0.652
 stale (>22d) 10570         0.597 0.544

calibration by score decile (1 = lowest risk):
        pages  mean_score  actual_decline_rate
decile                                        
1        3000       0.222                0.169
2        3000       0.407                0.460
3        3000       0.468                0.492
4        3000       0.523                0.537
5        3000       0.585                0.616
6        3000       0.639                0.665
7        3000       0.670                0.609
8        3000       0.699                0.653
9        3000       0.729                0.622
10       3000       0.797                0.599



permutation importance on one held-out fold (drop in AUC when the column is shuffled):
content_age_days          0.0647
search_volume             0.0069
impressions_90d           0.0023
main_intent               0.0000
content_type              0.0000
cpc                      -0.0002
competition              -0.0006
days_since_last_update   -0.0009
word_count               -0.0018


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.